In [9]:
%pip install pypdf
%pip install -U sentence-transformers


from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import glob
import os
import numpy as np



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
### READING AVAILABLE RESUME

resume_texts = []
resume_names = []

for file in glob.glob("../resources/resumes/*.pdf"):

    text = ""

    reader = PdfReader(file)

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + " "

    resume_texts.append(text)
    resume_names.append(os.path.basename(file))

print(len(resume_texts))

5000


In [6]:
### READING AVAILABLE JDs

reader = PdfReader("../resources/JD/20_Job_Descriptions.pdf")

jd_data = []

for page_num, page in enumerate(reader.pages, start=1):

    jd_text = page.extract_text()
    jd_role = f"JD_{page_num}"  # fallback

    if jd_text:
        lines = [line.strip() for line in jd_text.split("\n") if line.strip()]
        try:
            job_summary_idx = lines.index("Job Summary")
            if job_summary_idx > 0:
                jd_role = lines[job_summary_idx - 1]
        except ValueError:
            pass

    jd_data.append({
        "jd_pdf": "JD_Master.pdf",
        "jd_role": jd_role,
        "jd_page": page_num,
        "jd_id": f"JD_{page_num}",
        "text": jd_text
    })

jd_texts = [jd["text"] for jd in jd_data]

In [7]:
### CREATING MODEL

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
### JD EMBEDDINGS

jd_embeddings_file = "../resources/embeddings/jd_embeddings.npy"

if not os.path.exists(jd_embeddings_file):
    jd_embeddings = model.encode(
        jd_texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    os.makedirs("../resources/embeddings", exist_ok=True)
    np.save(jd_embeddings_file, jd_embeddings)
else:
    jd_embeddings = np.load(jd_embeddings_file)  

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
### RESUME EMBEDDINGS

resume_embeddings_file = "../resources/embeddings/resume_embeddings.npy"

if not os.path.exists(resume_embeddings_file):
    resume_embeddings = model.encode(
        resume_texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    os.makedirs("../resources/embeddings", exist_ok=True)
    np.save(resume_embeddings_file, resume_embeddings)
else:
    resume_embeddings = np.load(resume_embeddings_file)


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [18]:
### CREATING SIMILARITY MATRIX

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(
    jd_embeddings,
    resume_embeddings
)


print(jd_embeddings.shape)
print(resume_embeddings.shape)
print(similarity_matrix.shape)

(20, 384)
(5000, 384)
(20, 5000)


In [19]:
import numpy as np
import pandas as pd

rows = []

for jd_idx, jd in enumerate(jd_data):

    scores = similarity_matrix[jd_idx]

    top5_idx = np.argsort(scores)[::-1][:5]   ###argsort returns in ascending, hence [::-1]

    rows.append({
        "JD_ID": jd["jd_id"],
        "JD_Page": jd["jd_page"],
        "JD_Role": jd["jd_role"],
        "Best_Match_1": resume_names[top5_idx[0]],
        "Best_Match_2": resume_names[top5_idx[1]],
        "Best_Match_3": resume_names[top5_idx[2]],
        "Best_Match_4": resume_names[top5_idx[3]],
        "Best_Match_5": resume_names[top5_idx[4]]
    })

df = pd.DataFrame(rows)
df

,JD_ID,JD_Page,JD_Role,Best_Match_1,Best_Match_2,Best_Match_3,Best_Match_4,Best_Match_5
0,JD_1,1,Azure Data Engineer,Resume_033394_Steven_Watkins.pdf,Resume_034496_Susan_Watson.pdf,Resume_033028_Kaylee_Anderson.pdf,Resume_033674_Edgar_Watts.pdf,Resume_030521_Suzanne_Bowman.pdf
1,JD_2,2,Azure DevOps Engineer,Resume_032481_Christopher_Smith.pdf,Resume_034361_Brandon_Carter.pdf,Resume_032436_Derrick_Diaz.pdf,Resume_032194_Cindy_Ramirez.pdf,Resume_034359_John_Macias.pdf
2,JD_3,3,Data Scientist,Resume_034496_Susan_Watson.pdf,Resume_033674_Edgar_Watts.pdf,Resume_031631_Todd_Burgess.pdf,Resume_031929_Valerie_Martin.pdf,Resume_034893_Pamela_Swanson.pdf
3,JD_4,4,Data Analyst,Resume_034496_Susan_Watson.pdf,Resume_034175_Erin_Harmon.pdf,Resume_034893_Pamela_Swanson.pdf,Resume_033674_Edgar_Watts.pdf,Resume_031929_Valerie_Martin.pdf
4,JD_5,5,Full Stack Developer,Resume_030056_Christopher_Lin.pdf,Resume_034068_Jared_Cooley.pdf,Resume_033654_James_Fernandez.pdf,Resume_034378_Jason_Harrison.pdf,Resume_030599_Michelle_Long.pdf
5,JD_6,6,Python Developer,Resume_033674_Edgar_Watts.pdf,Resume_033778_Brian_Diaz.pdf,Resume_031356_Teresa_Deleon.pdf,Resume_033654_James_Fernandez.pdf,Resume_031758_Brittany_Moore.pdf
6,JD_7,7,Machine Learning Engineer,Resume_031609_Leah_Garcia.pdf,Resume_033674_Edgar_Watts.pdf,Resume_030089_Christopher_Lopez.pdf,Resume_034843_Kelli_Perez.pdf,Resume_034654_Deanna_Armstrong.pdf
7,JD_8,8,Generative AI Engineer,Resume_032436_Derrick_Diaz.pdf,Resume_034038_Michael_Lee.pdf,Resume_031609_Leah_Garcia.pdf,Resume_032076_Lisa_Nguyen.pdf,Resume_032194_Cindy_Ramirez.pdf
8,JD_9,9,Cloud Engineer,Resume_033826_Heather_Becker.pdf,Resume_032481_Christopher_Smith.pdf,Resume_031171_Christopher_Reese.pdf,Resume_032575_Kevin_Armstrong.pdf,Resume_030056_Christopher_Lin.pdf
9,JD_10,10,Site Reliability Engineer (SRE),Resume_032436_Derrick_Diaz.pdf,Resume_031917_Aaron_Wilson.pdf,Resume_033387_Joshua_Mitchell.pdf,Resume_034068_Jared_Cooley.pdf,Resume_031815_Jeffery_Gross.pdf


[{'jd_pdf': 'JD_Master.pdf',
  'jd_role': 'Azure Data Engineer',
  'jd_page': 1,
  'jd_id': 'JD_1',
  'text': "20 Professional Job Descriptions\nAzure Data Engineer\nJob Summary\nWe are seeking a skilled Azure Data Engineer to join our team and contribute to designing,\ndeveloping, and supporting business-critical solutions.\nKey Responsibilities\n\x7f Design, develop, and maintain enterprise solutions.\n\x7f Collaborate with cross-functional teams and stakeholders.\n\x7f Troubleshoot issues and optimize performance.\n\x7f Follow best practices for quality, security, and compliance.\n\x7f Prepare technical documentation and reports.\nRequired Qualifications\nBachelor's degree in a relevant field and 2+ years of professional experience.\nPreferred Skills\nProblem-solving, communication, teamwork, cloud technologies, automation, and data-driven\ndecision making.\n"},
 {'jd_pdf': 'JD_Master.pdf',
  'jd_role': 'Azure DevOps Engineer',
  'jd_page': 2,
  'jd_id': 'JD_2',
  'text': "Azure Dev